# GRUGRU-VAE + HydrAMP Hugging Face Demo

This notebook shows how to:
- load GRUGRU-VAE and HydrAMP from Hugging Face,
- load shared examples from the LAMP dataset,
- tokenize/prepare sequences,
- encode to latent space,
- decode/reconstruct and compare outputs side by side.

In [10]:
from __future__ import annotations

import torch
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [11]:
# ---- Edit these for your run ----
GRUGRU_MODEL_ID = "pszmk/lamp-grugru-vae"
# Hub remote-code must include encoder.encode (upload to this ref after API sync). Old tags lack it.
GRUGRU_MODEL_REVISION = "main"

HYDRAMP_MODEL_ID = "pszmk/hydramp"
HYDRAMP_MODEL_REVISION = "main"
HYDRAMP_TOKENIZER_ID = "pszmk/hydramp-aa-tokenizer"
HYDRAMP_TOKENIZER_REVISION = "main"

TOKENIZER_ID = "pszmk/protein-aa-fast-tokenizer"

DATASET_ID = "pszmk/LAMP-datasets"
DATASET_CONFIG = "nvidia_esm2_uniref_pretraining_data_leq50aa"
DATASET_SPLIT = "validation"
N_SAMPLES = 8
MAX_LENGTH = 64

## HydrAMP Hub vs local export

- **Private or gated repos**: set `HF_TOKEN` (see repository `.env-default`) or run `huggingface-cli login` before loading from the Hub.
- **Release uploads**: after you push with `--revision release/...` and `--tag`, set `HYDRAMP_MODEL_REVISION` / `HYDRAMP_TOKENIZER_REVISION` in the config cell to match (branch or tag).
- **Local export (no Hub)**: run `uv run python -m hydramp.export_to_hf --weights-dir src/hydramp/weights --local-export-dir ./path/to/model-export` and the tokenizer exporter with `--local-export-dir`; then set `HYDRAMP_LOCAL_MODEL_DIR` and `HYDRAMP_LOCAL_TOKENIZER_DIR` in the environment (or below) so the next cell loads them with `local_files_only=True`.

In [12]:
import os

grugru_model = AutoModel.from_pretrained(
    GRUGRU_MODEL_ID,
    revision=GRUGRU_MODEL_REVISION,
    trust_remote_code=True,
).to(device)
grugru_model.eval()

_hydramp_local = os.environ.get("HYDRAMP_LOCAL_MODEL_DIR")
if _hydramp_local:
    hydramp_model = AutoModel.from_pretrained(
        _hydramp_local,
        local_files_only=True,
        trust_remote_code=True,
    ).to(device)
else:
    hydramp_model = AutoModel.from_pretrained(
        HYDRAMP_MODEL_ID,
        revision=HYDRAMP_MODEL_REVISION,
        trust_remote_code=True,
    ).to(device)
hydramp_model.eval()

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

_hydramp_tok_local = os.environ.get("HYDRAMP_LOCAL_TOKENIZER_DIR")
if _hydramp_tok_local:
    hydramp_tokenizer = AutoTokenizer.from_pretrained(
        _hydramp_tok_local,
        local_files_only=True,
        trust_remote_code=True,
    )
else:
    hydramp_tokenizer = AutoTokenizer.from_pretrained(
        HYDRAMP_TOKENIZER_ID,
        revision=HYDRAMP_TOKENIZER_REVISION,
        trust_remote_code=True,
    )

dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT)

examples = [dataset[i]["sequence"] for i in range(min(N_SAMPLES, len(dataset)))]
examples[:3]

['VSGGRILMRRPGARISLITYWILVH',
 'MSQNVCPRPHPPDECSINEDCGMSMVCCLNDCGRRVCKPAYTPEHIIPRK',
 'RLSMRPTQEELEERNILK']

In [13]:
batch = tokenizer(
    examples,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
input_ids = batch["input_ids"].to(device)

# GRUGRU latent encoding (encode lives on encoder submodule).
grugru_mean, grugru_log_std = grugru_model.encoder.encode(input_ids)
grugru_z = grugru_mean

# HydrAMP latent encoding with HydrAMP-specific tokenizer.
hydramp_batch = hydramp_tokenizer(
    examples,
    padding="max_length",
    truncation=True,
    max_length=hydramp_model.config.sequence_length,
    return_tensors="pt",
)
hydramp_input_ids = hydramp_batch["input_ids"].to(device)
hydramp_mean, hydramp_log_std = hydramp_model.encoder.encode(hydramp_input_ids)
hydramp_z = hydramp_mean

print("GRUGRU input_ids shape:", tuple(input_ids.shape))
print("GRUGRU latent mean shape:", tuple(grugru_mean.shape))
print("HydrAMP input_ids shape:", tuple(hydramp_input_ids.shape))
print("HydrAMP latent mean shape:", tuple(hydramp_mean.shape))

AttributeError: 'GRUVAEEncoder' object has no attribute 'encode'

In [ ]:
# GRUGRU decode from latent positions (training-style path) + greedy reconstruction.
grugru_steps = input_ids.shape[1] - 1
grugru_decoder_out = grugru_model.decoder.forward_latent_positions(z=grugru_z, num_steps=grugru_steps)
grugru_logits = grugru_decoder_out.logits  # [B, T, V]
grugru_pred_token_ids = grugru_logits.argmax(dim=-1)

# HydrAMP: same GRUVAE-style API — forward_latent_positions -> .logits [B, T, V]; fixed T = config.sequence_length.
hydramp_logits = hydramp_model.forward_latent_positions(hydramp_z, return_logits=True).logits
hydramp_pred_token_ids = hydramp_logits.argmax(dim=-1)
hydramp_reconstructions = [
    hydramp_tokenizer.decode(ids, skip_special_tokens=True).replace(" ", "")
    for ids in hydramp_pred_token_ids.detach().cpu().tolist()
]

grugru_reconstructions = [
    tokenizer.decode(ids, skip_special_tokens=True).replace(" ", "")
    for ids in grugru_pred_token_ids.detach().cpu().tolist()
]

for i, (src, grugru_rec, hydramp_rec) in enumerate(
    zip(examples, grugru_reconstructions, hydramp_reconstructions, strict=False)
):
    print(f"[{i}] source : {src}")
    print(f"[{i}] grugru : {grugru_rec}")
    print(f"[{i}] hydramp: {hydramp_rec}")
    print("-" * 100)

In [ ]:
proteins = {
    "middle-1": ("FLYKWWIRIGRLKL", 4),
    "jurand-4": ("KYCRRFRWLTFRWL", 5),
    "jurand-2": ("KFRNRHRWKFKLIFRN", 5),
    "jurand-7": ("KKYWLIRKWIRLWFLT", 5),
    "mammuthusin-3": ("KTLKIIRLLF", 5),
    "hydrodamin-2": ("RMARNLVRYVQGLKKKKVI", 5),
}
# Reconstruct the proteins dictionary through both models
protein_names = list(proteins.keys())
protein_seqs = [seq for seq, _ in proteins.values()]

# Tokenize for GRUGRU
grugru_protein_batch = tokenizer(
    protein_seqs,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512,
)
grugru_protein_ids = grugru_protein_batch["input_ids"].to(device)

# Tokenize for HydrAMP (match fixed training length from model config)
hydramp_protein_batch = hydramp_tokenizer(
    protein_seqs,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=hydramp_model.config.sequence_length,
)
hydramp_protein_ids = hydramp_protein_batch["input_ids"].to(device)

# Encode with GRUGRU
grugru_protein_mean, grugru_protein_log_std = grugru_model.encoder.encode(grugru_protein_ids)
grugru_protein_z = grugru_protein_mean

# Encode with HydrAMP
hydramp_protein_mean, hydramp_protein_log_std = hydramp_model.encoder.encode(hydramp_protein_ids)
hydramp_protein_z = hydramp_protein_mean

# Decode with GRUGRU
grugru_protein_steps = grugru_protein_ids.shape[1] - 1
grugru_protein_decoder_out = grugru_model.decoder.forward_latent_positions(
    z=grugru_protein_z, num_steps=grugru_protein_steps
)
grugru_protein_logits = grugru_protein_decoder_out.logits
grugru_protein_pred_ids = grugru_protein_logits.argmax(dim=-1)

# Decode with HydrAMP (GRUVAE-style forward_latent_positions)
hydramp_protein_logits = hydramp_model.forward_latent_positions(hydramp_protein_z, return_logits=True).logits
hydramp_protein_pred_ids = hydramp_protein_logits.argmax(dim=-1)

# Convert to sequences
grugru_protein_reconstructions = [
    tokenizer.decode(ids, skip_special_tokens=True).replace(" ", "")
    for ids in grugru_protein_pred_ids.detach().cpu().tolist()
]

hydramp_protein_reconstructions = [
    hydramp_tokenizer.decode(ids, skip_special_tokens=True).replace(" ", "")
    for ids in hydramp_protein_pred_ids.detach().cpu().tolist()
]

# Display results
for name, src, grugru_rec, hydramp_rec in zip(
    protein_names, protein_seqs, grugru_protein_reconstructions, hydramp_protein_reconstructions, strict=False
):
    print(f"{name}:")
    print(f"  source : {src}")
    print(f"  grugru : {grugru_rec}")
    print(f"  hydramp: {hydramp_rec}")
    print("-" * 100)